# Evaluer la qualité de votre préparation de données et de votre Modèle

---



## Installer DeepFix

Accéder à la documentation de DeepFix: [link text](https://delcaux-labs.github.io/deepfix/)

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

In [ ]:
!uv pip install git+https://github.com/delcaux-labs/deepfix.git

## **Re-démarrer votre session Colab avant de continuer!**
``Runtime > 'Restart session'``
Ceci est dû à un problème de conflit entre différentes version de *numpy*

### Charger les donnees

In [1]:
from deepfix_sdk import DeepFixClient
import os

In [ ]:
os.environ["DEEPFIX_API_KEY"] = ... # Demander une API key at https://deepfix.delcaux.com

In [ ]:
client = DeepFixClient(timeout=120)

In [4]:
from deepfix_sdk.data.datasets import TabularDataset
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [ ]:
# Chargez les données
X,y = load_breast_cancer(as_frame=True,return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
dataset_name = "breast_cancer_classification"

label = "target"
train = X_train.copy()
train[label] = y_train
cat_features = X_train.select_dtypes(include=['object','string','category']).columns.tolist()

test = X_test.copy()
test[label] = y_test

In [ ]:
# Creer les datasets
train_data = TabularDataset(dataset=train, dataset_name=dataset_name, label=label, cat_features=cat_features)
val_data = TabularDataset(dataset=test, dataset_name=dataset_name, label=label, cat_features=cat_features)

In [ ]:
# train_data.data.head()

### Evaluer la qualité de votre préparation de données

In [ ]:
# Ingestion des données
result = client.get_diagnosis(
    train_data=train_data,
    test_data=val_data,
    language="french",
)

In [10]:
# Résultats
result.to_text(False)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ L'analyse consolidée révèle un dataset avec une qualité de base excellente mais des problèmes structurels            │
│ importants. Trois caractéristiques présentent des risques de fuite de données avec des scores de pouvoir prédictif   │
│ anormalement élevés (>0.7), nécessitant une investigation immédiate. De plus, une multicolinéarité sévère affecte    │
│ 22+ paires de caractéristiques, ce qui pourrait compromettre la stabilité et l'interprétabilité du modèle.           │
│ Cependant, l'intégrité des données est impeccable avec aucune duplication, conflit d'étiquettes ou problèmes de      │
│ nullité détectés. La distribution entre les ensembles d'entraînement et de test est également cohérente. Les         │
│ priorités d'action incluent l'investigation des caractéristiques suspectées de fuite de données et la réduction de   │
│ la multicolinéarité par sélection de caractéristiques.                                                               │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  3                                                            
 Severity Distribution           HIGH: 1  MEDIUM: 1  LOW: 1                                   

                                  HIGH Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Scores de pouvoir prédictif élevés       │ Investiger la source et le calcul des    │
│     │ indiquant des risques potentiels de      │ caractéristiques 'worst perimeter',      │
│     │ fuite de données                         │ 'worst concave points' et 'worst radius' │
│     │ Evidence: Les caractéristiques 'worst    │ pour s'assurer qu'elles ne contiennent   │
│     │ perimeter' (PPS 0.77), 'worst concave    │ pas d'informations directes ou           │
│     │ points' (PPS 0.75), et 'worst radius'    │ indirectes sur la variable cible         │
│     │ (PPS 0.71) dépassent le seuil de 0.7     │ Les valeurs PPS supérieures à 0.7        │
│     │ dans les données d'entraînement          │ suggèrent que ces caractéristiques       │
│     │                                          │ peuvent être trop prédictives, indiquant │
│     │                                          │ potentiellement une fuite de données ou  │
│     │                                          │ des caractéristiques trop étroitement    │
│     │                                          │ liées à la variable cible                │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (1)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Multicolinéarité sévère avec plus de 22  │ Appliquer des techniques de sélection de │
│     │ paires de caractéristiques montrant une  │ caractéristiques ou de réduction de      │
│     │ corrélation > 0.9                        │ dimensionnalité pour éliminer les        │
│     │ Evidence: Plusieurs paires fortement     │ caractéristiques redondantes et          │
│     │ corrélées incluant ('mean radius',       │ améliorer la stabilité du modèle         │
│     │ 'worst radius'), ('mean perimeter',      │ Une multicolinéarité élevée peut causer  │
│     │ 'worst perimeter'), ('mean area', 'worst │ des coefficients de modèle instables, du │
│     │ area') avec une corrélation > 0.9        │ sur-apprentissage et des difficultés     │
│     │                                          │ dans l'interprétation de l'importance    │
│     │                                          │ des caractéristiques                     │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                   LOW Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Excellente intégrité des données sans    │ Maintenir les pratiques actuelles de     │
│     │ problèmes de qualité détectés            │ collecte et de prétraitement des données │
│     │ Evidence: Tous les contrôles d'intégrité │ pour préserver la qualité des données    │
│     │ des données ont été passés : 0% de       │ Le dataset montre une excellente         │
│     │ données dupliquées, 0% d'étiquettes      │ intégrité sans problèmes de qualité      │
│     │ conflictuelles, pas de types de données  │ détectés, ce qui donne confiance dans la │
│     │ mélangés, pas de problèmes de caractères │ fiabilité des données                    │
│     │ spéciaux                                 │                                          │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''

### Evaluer la qualité de votre modèle

In [11]:
# Fit model
model_name = "HistGradientBoostingClassifier"
clf = HistGradientBoostingClassifier(max_depth=3)
clf = clf.fit(train_data.X, train_data.y)

In [ ]:
# Ingestion des données
result = client.get_diagnosis(
    train_data=train_data,
    test_data=val_data,
    model_name=model_name,
    model=clf,
    language="french",
)

In [13]:
# Résultats
result.to_text(False)

╭──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                               DEEPFIX ANALYSIS RESULT                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────────── Summary ───────────────────────────────────────────────────────╮
│ L'analyse croisée des artefacts révèle un modèle aux performances exceptionnelles (AUC 0.99-1.0) mais avec des       │
│ opportunités significatives d'amélioration. La qualité des données est excellente avec une dérive minimale, mais des │
│ redondances de caractéristiques et une sous-utilisation de caractéristiques à variance élevée sont identifiées. Le   │
│ principal défi réside dans le manque critique d'informations contextuelles et de documentation, couplé à des         │
│ configurations de régularisation potentiellement sous-optimales. Des actions correctives sur la documentation, la    │
│ configuration du modèle et l'optimisation des caractéristiques sont recommandées pour garantir la robustesse à long  │
│ terme et la facilité d'utilisation du modèle.                                                                        │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                      Summary Statistics                                      
 Metric                          Value                                                        
 Total Findings                  4                                                            
 Severity Distribution           MEDIUM: 3  HIGH: 1                                           

                                  HIGH Severity Issues (1)                                   
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Manque critique d'informations           │ Documenter complètement les métadonnées  │
│     │ contextuelles et de configuration de     │ du modèle et réviser les paramètres de   │
│     │ régularisation dans le modèle            │ régularisation et de feuilles            │
│     │ Evidence: Absence d'informations sur la  │ L'absence d'informations contextuelles   │
│     │ taille du jeu de données, métriques de   │ empêche l'évaluation de l'adéquation du  │
│     │ performance, et régularisation L2=0.0    │ modèle, et le manque de régularisation   │
│     │ avec min_samples_leaf=20 potentiellement │ pourrait causer du surapprentissage à    │
│     │ restrictif                               │ long terme                               │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

                                 MEDIUM Severity Issues (3)                                  
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Finding                                  ┃ Action                                   ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ Échec de l'analyse des artefacts de jeu  │ Corriger l'implémentation de l'analyseur │
│     │ de données dû à une erreur technique     │ DatasetArtifactsAnalyzer et réexécuter   │
│     │ Evidence: Erreur AttributeError:         │ l'analyse                                │
│     │ 'DatasetArtifacts' object has no         │ L'absence d'analyse des métadonnées du   │
│     │ attribute 'statistics' lors de la        │ jeu de données limite la compréhension   │
│     │ construction du prompt                   │ complète du contexte des données         │
│ 2   │ Redondance des caractéristiques avec     │ Appliquer une réduction de               │
│     │ corrélations élevées et caractéristiques │ dimensionnalité (PCA) et réviser         │
│     │ sous-utilisées malgré une qualité de     │ l'importance des caractéristiques tout   │
│     │ données exceptionnelle                   │ en maintenant les pratiques de           │
│     │ Evidence: 27 paires de caractéristiques  │ validation actuelles                     │
│     │ avec corrélation >0.9, 16                │ L'élimination de la redondance et une    │
│     │ caractéristiques à variance élevée non   │ meilleure utilisation des                │
│     │ utilisées, AUC 0.99-1.0, dérive minimale │ caractéristiques à variance élevée       │
│     │ (0.15 multivariée, 0.12 prédiction)      │ pourraient améliorer encore les          │
│     │                                          │ performances malgré les excellents       │
│     │                                          │ résultats actuels                        │
│ 3   │ Configuration ambiguë des                │ Documenter le traitement des             │
│     │ caractéristiques catégorielles et        │ caractéristiques catégorielles et        │
│     │ résultats d'arrêt anticipé manquants     │ inclure les résultats de l'arrêt         │
│     │ Evidence:                                │ anticipé                                 │
│     │ categorical_features='from_dtype' sans   │ La clarté sur le prétraitement et la     │
│     │ documentation, early_stopping='auto'     │ convergence du modèle est essentielle    │
│     │ sans résultats de validation             │ pour une utilisation et un déploiement   │
│     │                                          │ corrects                                 │
└─────┴──────────────────────────────────────────┴──────────────────────────────────────────┘

''

# A votre tour



1.   Chargez vos données
2.   Evaluer la qualité de votre préparation des données
3.   Evaluer la performance de votre modèle

